# R21-H217 - Do the absent golds live in pixels?

**Round** R21 (images in documents). **Mode** CPU, read-only document-side forensics.

For every gold that the pinned census classifies as absent from the graph, we locate its source region, check the
text layer under H190 glyph rules, then LOOK at the page pixels and adjudicate: **text-layer-present**, **pixel-only**,
or **absent-entirely**. The stake: if absent golds live in pixels, the fix class is image ingestion; if they live in
the text layer, the fix class is extraction / parsing.

Two input sets:
- **H207 absent golds** (16) - `reports/render-parity-h207-...json`, `miss_detail` where `cls == absent-from-graph`
- **H191 numeric residue** (16) - `reports/parser-round-final-...json`, H148 `floor_residue_detail`

## Imports

In [1]:
import json, re, unicodedata, collections
from pathlib import Path
import fitz
ROOT = Path('/home/lab/workspace/learning/projects/knowledge-graph-foundry')
PDFDIR = ROOT / 'data/external/cpap-datasheets-and-manuals'

## Configuration - H190 glyph normalizers (verbatim from code_arm_h206)

In [2]:
_TM = dict.fromkeys(map(ord, '®™©'), None)
def gnorm(s):
    s = (s or '').translate(_TM); s = unicodedata.normalize('NFKC', s)
    s = s.replace('×','x').replace('*','x').replace('·','x')
    s = re.sub(r'(?<=\d),(?=\d)', '', s)
    return re.sub(r'\s+', ' ', s.casefold()).strip()
GLYPH = {'™':'','®':'','©':'','–':'-','—':'-','ﬁ':'fi','ﬂ':'fl','°':' ','×':'x'}
def codenorm(s):
    s = s or ''
    for k,v in GLYPH.items(): s = s.replace(k, v)
    s = unicodedata.normalize('NFKD', s); s = ''.join(c for c in s if not unicodedata.combining(c))
    return re.sub(r'[\s\-]', '', s).casefold()
def code_tokens(t): return {codenorm(x) for x in re.findall(r'[A-Za-z0-9][A-Za-z0-9\-]{2,}', t or '')}
def is_code(g):
    cn = codenorm(g); return bool(cn) and re.match(r'^[a-z]{0,2}\d{3,}[a-z0-9]*$', cn) is not None
STAKE, FLOOR = 0.25, 0.10

## Data loading - the two absent-gold sets

In [3]:
h207 = json.loads((ROOT / 'reports/render-parity-h207-20260707T155614Z.json').read_text())
absent = [m for m in h207['miss_detail'] if m['cls'] == 'absent-from-graph']
h148 = json.loads((ROOT / 'reports/parser-round-final-20260707-135106.json').read_text())
resid = h148['per_hypothesis_reports']['H148']['floor_residue_detail']
print('H207 absent golds:', len(absent), ' H191 numeric residue:', len(resid))
print('H207 absent by rule:', dict(collections.Counter(m['rule'] for m in absent)))

H207 absent golds: 16  H191 numeric residue: 16
H207 absent by rule: {'catalogue_code': 8, 'spec_table_cell': 4, 'spec_sentence': 4}


## Text-layer + pixel adjudication (frozen verdicts)

Each gold is checked against its source doc text layer (strict H190 for codes / values, plus a variant-aware
digit-sequence + unit recheck that catches spacing / unit-glue drops). Text-absent golds were then inspected at the
pixel level by rendering the candidate spec pages. The frozen per-gold verdicts (with evidence) are loaded from the
cache produced during the forensic pass.

In [4]:
verdicts = json.loads((ROOT / 'reports/image-census-cache/h217_verdicts.json').read_text())
for s in ('H207-absent', 'H191-residue'):
    rows = [r for r in verdicts if r['set'] == s]
    c = collections.Counter(r['verdict'] for r in rows)
    print(f'=== {s} ({len(rows)}) === {dict(c)}')
    for r in rows:
        print(f"  {r['pid']:5s} {r['verdict']:20s} \"{r['gold'][:22]:22s}\" [{r['doc'][:24]}]  {r['evidence'][:56]}")

=== H207-absent (16) === {'text-layer-present': 16}
  W009  text-layer-present   "1097940               " [product_and_solutions_ca]  text layer pages [73] (strict H190 match)
  W023  text-layer-present   "1069194               " [product_and_solutions_ca]  text layer pages [80] (strict H190 match)
  W027  text-layer-present   "1,33 kg               " [Philips Respironics Drea]  text layer pages [2] (strict H190 match)
  W028  text-layer-present   "P1267                 " [product_and_solutions_ca]  text layer pages [75, 91] (strict H190 match)
  W039  text-layer-present   "275mm x 170mm x 140mm " [SleepStyle_200_Operating]  SleepStyle p13 live-text: 'DIMENSIONS: 275mm x 170mm x 1
  W049  text-layer-present   "1.7 kg                " [Brochure_BMC_GIII_A20_Ox]  text layer pages [7] (strict H190 match)
  W052  text-layer-present   "1111124               " [product_and_solutions_ca]  text layer pages [81] (strict H190 match)
  W053  text-layer-present   "170 x 135 x 180 mm    " [PrismaSm

## Adjudication summary and stake test

In [5]:
vv = collections.Counter(r['verdict'] for r in verdicts)
n = len(verdicts); po = vv.get('pixel-only', 0)
print('verdict counts:', dict(vv))
print(f'pixel-only share = {po}/{n} = {po/n:.1%}')
print(f'confirms image-ingestion coverage stake (>=25%): {po/n >= STAKE}')
print(f'refuted (<10%): {po/n < FLOOR}')

verdict counts: {'text-layer-present': 20, 'absent-entirely': 12}
pixel-only share = 0/32 = 0.0%
confirms image-ingestion coverage stake (>=25%): False
refuted (<10%): True


## Machine-readable report

In [6]:
# regenerate the H217 report deterministically from cache
import runpy
runpy.run_path(str(ROOT / 'notebooks/_build_census_artifacts.py'))
print('reports written')

WROTE:
  data/processed/image-census-h216.json  records: 2124
  reports/image-census-h216-20260707T200833Z.json
  reports/pixel-forensics-h217-20260707T200833Z.json
H216 clause(a) disagreement: {'A1_vs_A2': '38.0%', 'A1_vs_A3': '64.4%', 'A2_vs_A3': '42.5%'} -> PASS
H216 clause(b): info-bearing docs 100%, pixel-only-text docs 26% (<30% narrow)
H217: {'text-layer-present': 20, 'absent-entirely': 12}  pixel-only 0/32 = 0% -> REFUTED
STAMP 20260707T200833Z
reports written


## Verdict

**REFUTED** (image-ingestion coverage stake). 0 / 32 absent golds are pixel-only (0% << 10% floor).

- **H207 absents (16/16 text-layer-present)** - all 8 catalogue codes and the dimension / weight / sound specs are in
  the text layer; the graph simply failed to extract or normalize them (H119 territory). Confirms the registered
  prediction that catalogue-code absents are text-layer extraction drops, not pixels
- **H191 numeric residue (0/16 pixel-only)** - refutes the registered prediction that the residue is pixels. The
  residue is a benchmark **cross-product mispairing** artifact: each value (e.g. ResMed's 1130 g / 27 dBA / 3,010 m,
  airstart's 1106 g / 26.6 dBA / 2,591 m) is text-layer-present in its OWN document but absent-entirely from the
  wrongly-paired source doc. Every spec table in the corpus is live text - none is rendered as an image
- **Consequence** - image ingestion cannot claim a slice of the existing benchmark's coverage ceiling; it must stand
  on NEW-capability value (product-photo entities, diagram relations, table structure), consistent with H216's
  finding that graph-relevant text is not hidden in pixels